<div style="background-color:#0f172a; padding:15px; border-radius:10px; color:white;">
<h2>📌 Project Overview </h2>
<p><b>Objective:</b> Build a reasoning model pipeline for the NVIDIA Nemotron Challenge.</p>

<p><b>Techniques Used:</b></p>
<ul>
<li>Data Loading & Inspection</li>
<li>Text Preprocessing</li>
<li>Feature Engineering</li>
<li>Transformer-based Modeling (or baseline ML)</li>
<li>Evaluation Metrics</li>
<li>Submission Generation</li>
</ul>

<p>This notebook is structured in modular slots for clarity and scalability.</p>
</div>

<h2 style='color:#2563eb;'>🔹 SLOT 1: Load Dataset</h2>

In [2]:
import pandas as pd
import os

# Kaggle dataset path
DATA_PATH = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge"

# List files
for dirname, _, filenames in os.walk(DATA_PATH):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Example loading (adjust filenames accordingly)
try:
    train = pd.read_csv(f"{DATA_PATH}/test.csv")
    test = pd.read_csv(f"{DATA_PATH}/train.csv")
    print("✅ Dataset Loaded Successfully")
except:
    print("⚠️ Adjust file names according to dataset")

train.head(5)

/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv
/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv
✅ Dataset Loaded Successfully


,id,prompt
0,00066667,"In Alice's Wonderland, a secret bit manipulati..."
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati..."
2,00189f6a,"In Alice's Wonderland, secret encryption rules..."


In [4]:
print("Train Shape :",train.shape)
print("Test Shape :",test.shape)

train.info()
train.describe()

Train Shape : (3, 2)
Test Shape : (9500, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      3 non-null      object
 1   prompt  3 non-null      object
dtypes: object(2)
memory usage: 180.0+ bytes


,id,prompt
count,3,3
unique,3,3
top,00066667,"In Alice's Wonderland, a secret bit manipulati..."
freq,1,1


# Preprocessing

In [5]:
import re

def clean_text(text):
    text =str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9 ]", "", text)
    return text 

if 'text' in train.columns:
    train['clean_text'] = train['text'].apply(clean_text)

train.head()


,id,prompt
0,00066667,"In Alice's Wonderland, a secret bit manipulati..."
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati..."
2,00189f6a,"In Alice's Wonderland, secret encryption rules..."


# Feature Engineering

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=500)

if 'clean_text' in train.columns:
    X = vectorizer.fit_transform(train['clean_text'])
else:
    X = None

print("Feature Matrix Shape :",X.shape if X is not  None else "N/A")

Feature Matrix Shape : N/A


# Model Training

In [16]:
print("Columns in train:", train.columns)

# Step 1: Ensure prompt column exists
if 'prompt' not in train.columns:
    print("❌ 'prompt' column not found.")
else:
    from sklearn.feature_extraction.text import TfidfVectorizer

    # Convert prompt → features
    vectorizer = TfidfVectorizer(max_features=500)
    X = vectorizer.fit_transform(train['prompt'].astype(str))

    print("✅ Feature matrix created:", X.shape)

    # Since no labels → we simulate clustering instead
    from sklearn.cluster import KMeans

    model = KMeans(n_clusters=3, random_state=42)
    model.fit(X)

    print("✅ Unsupervised Model (KMeans) Trained")

Columns in train: Index(['id', 'prompt'], dtype='object')
✅ Feature matrix created: (3, 114)
✅ Unsupervised Model (KMeans) Trained


# Evaluation

In [18]:

try:
    print("ℹ️ No labels available → skipping accuracy.")

    # Show cluster distribution
    import numpy as np
    labels = model.labels_

    print("Cluster Distribution:")
    print(np.bincount(labels))

except Exception as e:
    print("❌ Evaluation Failed:", e)



ℹ️ No labels available → skipping accuracy.
Cluster Distribution:
[1 1 1]


In [19]:
import zipfile

try:
    # Step 1: Transform test data
    X_test = vectorizer.transform(test['prompt'].astype(str))

    # Step 2: Predict
    preds = model.predict(X_test)

    # Step 3: Create CSV
    submission = pd.DataFrame({
        "id": test['id'],
        "cluster": preds
    })

    submission.to_csv("submission.csv", index=False)
    print("✅ submission.csv created")

    # Step 4: Create ZIP file
    with zipfile.ZipFile("submission.zip", "w") as z:
        z.write("submission.csv")

    print("✅ submission.zip created successfully")

except Exception as e:
    print("❌ Submission Failed:", e)

✅ submission.csv created
✅ submission.zip created successfully
